# recs_030 — Stage 4 explanation quality: reference-free heuristics

`recs_029` qualitatively spot-checked `generate_explanation()` on 5 examples. This notebook
runs it over the full 200-example `val_llm_mini_v1` cohort and scores each explanation with
cheap, reference-free heuristics (`steam_review_ml.evaluation.explanation_heuristics`) as a
sanity pass before building the real LLM-judge harness (Track B, `docs/plans/rag_extension_plan.md`).

Two proxies, mirroring the judge rubric we'll use later:
- **Groundedness proxy**: content-word overlap between the explanation and the recommended
  game's own IGDB text, plus a tag-leakage check against IGDB's closed genre/theme vocabulary.
- **Relevance proxy**: embedding cosine similarity between the explanation and the query text.

No promotion bar here -- this just catches obviously broken/hallucinating explanations cheaply,
before spending real Anthropic API calls on the judge.

Uses `RAGRecommender` (the currently shipped production pipeline, not the old `StackedRecommender`
used in `recs_029` before the retrieval-stage rename/pivot).


In [1]:
from pathlib import Path

import pandas as pd

from steam_review_ml.evaluation.explanation_eval_pipeline import generate_or_load_explanations
from steam_review_ml.recommender.rag_recommender import RAGRecommender

REPO_ROOT = Path.cwd().parent.parent
GGUF_PATH = REPO_ROOT / "artifacts/models/llm_local/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf"
EXPLANATIONS_CACHE = REPO_ROOT / "artifacts/recs/eval_cache/val_llm_mini_v1/explanations.parquet"

rec = RAGRecommender.from_serve_config(repo_root=REPO_ROOT)
print(f"method_id={rec.method_id}, k_retrieval={rec.k_retrieval}, k_final={rec.k_final}")

# Same source as rec_app_name below -- keeps both names looked up consistently.
app_name_by_id = dict(zip(rec.retriever.index_frame["app_id"], rec.retriever.index_frame["app_name"]))


/home/ryanr/miniconda3/envs/tf_condaforge/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-08-26 13:18:02.982335: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-26 13:18:03.008058: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787764683.036619 1768483 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787764683.045791 1768483 cuda_bl

method_id=two_tower_v1_v2a_embed_query_logpop_blend, k_retrieval=100, k_final=10


## Generate top-1 recommendation + explanation for the full cohort

Cached to `explanations.parquet` -- rerunning the notebook won't re-pay the local-LLM cost
unless the cache is deleted.


In [ ]:
COHORT_PATH = REPO_ROOT / "artifacts/recs/eval_cache/val_llm_mini_v1/example_cohort.parquet"
cohort_df = pd.read_parquet(COHORT_PATH)[["query_app_id", "query_text"]]

results_df = generate_or_load_explanations(
    rec,
    cohort_df,
    app_name_by_id,
    cache_path=EXPLANATIONS_CACHE,
    gguf_path=GGUF_PATH,
)
results_df.head()


## Score heuristics

Groundedness proxy (content-word overlap + tag leakage) and relevance proxy (embedding
similarity to the query text).


In [ ]:
from sentence_transformers import SentenceTransformer

from steam_review_ml.evaluation.explanation_eval_pipeline import score_explanations

embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
scored_df = score_explanations(results_df, embed_model)
scored_df.head()


## Summary

In [4]:
print(f"n examples: {len(scored_df)}")
print(f"degenerate output: {scored_df['is_degenerate'].mean():.1%}")
print(f"any ungrounded tag flagged: {(scored_df['ungrounded_tags'].str.len() > 0).mean():.1%}")
print(f"content_overlap_ratio: mean={scored_df['content_overlap_ratio'].mean():.3f}, "
      f"median={scored_df['content_overlap_ratio'].median():.3f}")
print(f"relevance_cosine: mean={scored_df['relevance_cosine'].mean():.3f}, "
      f"median={scored_df['relevance_cosine'].median():.3f}")


n examples: 200
degenerate output: 0.0%
any ungrounded tag flagged: 23.5%
content_overlap_ratio: mean=0.181, median=0.152
relevance_cosine: mean=0.595, median=0.602


## Worst examples (lowest groundedness overlap, most ungrounded tags)

Manual eyeball check -- these are the ones a prompt tweak should target first.


In [5]:
worst = scored_df.sort_values("content_overlap_ratio").head(10)
for row in worst.to_dict("records"):
    print(f"query: {row['query_app_name']} (app_id={row['query_app_id']}) "
          f"-> rec: {row['rec_app_name']} (app_id={row['rec_app_id']}), "
          f"overlap={row['content_overlap_ratio']:.2f}, "
          f"ungrounded_tags={row['ungrounded_tags']}")
    print(f"  Explanation: {row['explanation']}")
    print()


query: Darkest Dungeon® (app_id=262060) -> rec: Terraria (app_id=105600), overlap=0.04, ungrounded_tags=[]
  Explanation: We think you'll love Terraria because it offers a similar sense of exploration and survival, where you must navigate the unknown to uncover hidden treasures and overcome challenges. Like Darkest Dungeon, Terraria also involves managing risks and making tough decisions - in this case, deciding how to allocate your time between digging for resources, building your base, and battling fierce enemies. This mix of discovery, strategy, and action should appeal to fans of Darkest Dungeon's gothic roguelike gameplay.

query: Among Us (app_id=945360) -> rec: Terraria (app_id=105600), overlap=0.05, ungrounded_tags=[]
  Explanation: We think you'll love Terraria because, like Among Us, it's a game that encourages exploration and strategy while also having an element of danger lurking beneath the surface. In Terraria, you'll need to be careful as you dig and build, as you never 